# Exercise 5 · Interactive AI Prototyping with Gradio
This chatbot uses TF-IDF similarity to retrieve answers from a small set of Python lab notes. It runs locally without an API key and uses chat history to answer follow-up requests for examples.

Run the source cell, then open the local URL printed by Gradio. Select a sample prompt or type your own message. To stop the app, run demo.close() or shut down the notebook kernel.

In [ ]:
!pip install gradio==6.32.0 scikit-learn

In [2]:
import re

import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer

examples = [
    "How do I calculate the mean across each row?",
    "How do I blend two images with OpenCV?",
    "What does a word cloud show?",
]

knowledge = [
    (
        "pandas dataframe mean average row columns calculate f5",
        "Use `df[['f1', 'f2', 'f3', 'f4']].mean(axis=1)`. "
        "`axis=1` means Pandas averages across the selected columns for each row.",
        "For a row containing 10, 20, 30, and 40, the mean is (10 + 20 + 30 + 40) ÷ 4 = 25.",
    ),
    (
        "opencv cv2 addWeighted image blend blending two images resize",
        "Resize the images to the same dimensions, then call "
        "`cv2.addWeighted(img1, 0.6, img2, 0.4, 0)`. "
        "The output combines 60% of the first image with 40% of the second.",
        "If a channel has value 100 in the first image and 200 in the second, "
        "its blended value is 0.6 × 100 + 0.4 × 200 = 140.",
    ),
    (
        "word cloud wordcloud text summaries feedback frequency show",
        "A word cloud uses larger text for more frequent terms. "
        "Remove common stop words before creating it from the review summaries.",
        "If “quality” appears 80 times and “delivery” appears 25 times, "
        "“quality” will generally appear larger in the cloud.",
    ),
]

vectorizer = TfidfVectorizer(stop_words="english")
topic_vectors = vectorizer.fit_transform([entry[0] for entry in knowledge])


def match_topic(text):
    scores = (vectorizer.transform([text]) @ topic_vectors.T).toarray()[0]
    best = scores.argmax()
    return int(best) if scores[best] > 0 else None


def reply(message, history):
    text = message.strip().lower()
    if re.fullmatch(r"(hi|hello|hey)[!. ]*", text):
        return "Hello! Pick a sample prompt or ask about Pandas, OpenCV, or word clouds."

    topic = match_topic(text)
    wants_example = bool(re.search(r"\bexamples?\b", text))
    if wants_example and topic is None:
        for turn in reversed(history):
            if turn["role"] != "user":
                continue
            content = turn["content"]
            if isinstance(content, dict):
                content = content.get("text", "")
            elif isinstance(content, list):
                content = " ".join(
                    part if isinstance(part, str) else part.get("text", "")
                    for part in content
                    if isinstance(part, (dict, str))
                )
            if isinstance(content, str):
                topic = match_topic(content)
                if topic is not None:
                    break

    if topic is None:
        return "Try asking about row means, OpenCV image blending, or word clouds."
    return knowledge[topic][2 if wants_example else 1]


demo = gr.ChatInterface(
    fn=reply,
    title="Python Lab Chatbot",
    description=(
        "Welcome! Ask about Pandas, OpenCV, or word clouds. "
        "Select a sample prompt to begin, then try “Give me an example.” "
        "This local demo retrieves answers from a small set of lab notes."
    ),
    examples=examples,
)

# Keep this notebook open while using the local Gradio app.
demo.launch(share=False, inline=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
demo.close()

Closing server running on port: 7860
